## 面试问题

Backtrack/换策略：失败后回退到哪个检查点、换什么策略？

## 回答主线

一步失败后不该死磕同策略(退化 stall)也不该整个放弃(premature)，而应 backtrack 到检查点并换策略。本 Notebook 用检索任务：关键词检索(字面匹配)对语义查询无命中，回退检索分叉点、换语义检索命中。对比死磕关键词(重复失败)与 backtrack 换策略(成功)。

## 真实案例

查询「忘记登录口令怎么办」与文档语义相关但无字面重叠。关键词检索无命中，语义检索(同义词扩展)命中。策略候选 `[keyword, semantic]`。数据为教学检索，不代表真实检索系统。

In [1]:
documents = ["如何重置密码", "账户安全设置指南"]  # 教学文档集合。
query = "忘记登录口令怎么办"  # 用户查询与文档语义相关但无字面重叠。

def keyword_search(query, docs):  # 关键词检索：要求字面字符重叠。
    hits = [d for d in docs if any(ch in d for ch in query)]  # 按单字重叠匹配。
    return hits  # 返回命中文档。

def semantic_search(query, docs):  # 语义检索：用同义词映射扩展后匹配。
    synonyms = {"口令": "密码", "登录": "账户"}  # 简化的同义词表。
    expanded = query  # 从原查询开始扩展。
    for k, v in synonyms.items():  # 用同义词替换扩展查询。
        expanded = expanded.replace(k, v)  # 扩展查询词。
    hits = [d for d in docs if any(ch in d for ch in expanded)]  # 用扩展查询匹配。
    return hits  # 返回命中文档。

print("查询:", query)  # 展示查询。
print("关键词检索命中:", keyword_search(query, documents))  # 展示关键词检索无命中。
print("语义检索命中:", semantic_search(query, documents))  # 展示语义检索有命中。

查询: 忘记登录口令怎么办
关键词检索命中: []
语义检索命中: ['如何重置密码', '账户安全设置指南']


## 基线（Baseline）

反面基线：死磕同一策略。关键词检索无命中却重复三次，始终 not_found，退化成无进展。

In [2]:
def run_retry_same(query, docs, strategy, max_retries=3):  # 死磕同一策略重复检索。
    attempts = []  # 记录每次尝试。
    for _ in range(max_retries):  # 重复尝试同一策略。
        hits = strategy(query, docs)  # 执行检索。
        attempts.append(len(hits))  # 记录命中数。
        if hits:  # 有命中则成功。
            return {"status": "found", "attempts": attempts, "hits": hits}  # 返回成功。
    return {"status": "not_found", "attempts": attempts}  # 全部失败。

retry_result = run_retry_same(query, documents, keyword_search)  # 死磕关键词检索。
print("死磕关键词检索:", retry_result)  # 展示重复三次都无命中。

死磕关键词检索: {'status': 'not_found', 'attempts': [0, 0, 0]}


## 失败案例与修正

死磕退化成无进展。修正是 backtrack：在检索分叉点打检查点，从策略候选里排除已失败的策略、换下一个。keyword 失败后换 semantic 成功。

In [3]:
STRATEGY_CANDIDATES = [("keyword", keyword_search), ("semantic", semantic_search)]  # 策略候选列表。

def make_checkpoint(query, docs, tried):  # 构造一个可回退的检查点。
    return {"query": query, "docs": docs, "tried": list(tried)}  # 保存查询与已尝试策略。

def run_backtrack(checkpoint, candidates):  # 失败后回退检查点并换策略。
    tried = list(checkpoint["tried"])  # 从检查点恢复已尝试策略。
    log = []  # 记录换策略过程。
    for name, strategy in candidates:  # 遍历策略候选。
        if name in tried:  # 跳过已失败策略。
            continue  # 换下一个。
        hits = strategy(checkpoint["query"], checkpoint["docs"])  # 执行当前策略。
        tried.append(name)  # 标记已尝试。
        log.append((name, len(hits)))  # 记录该策略命中数。
        if hits:  # 有命中则成功。
            return {"status": "found", "strategy": name, "log": log, "hits": hits}  # 返回成功策略。
    return {"status": "not_found", "log": log}  # 全部候选失败。

In [4]:
checkpoint = make_checkpoint(query, documents, [])  # 在检索分叉点打检查点。
backtrack_result = run_backtrack(checkpoint, STRATEGY_CANDIDATES)  # 回退换策略检索。
print("检查点:", {"query": checkpoint["query"], "tried": checkpoint["tried"]})  # 展示初始检查点。
print("backtrack 换策略:", backtrack_result["status"], "最终策略", backtrack_result.get("strategy"))  # 展示 keyword 失败后换 semantic 成功。

检查点: {'query': '忘记登录口令怎么办', 'tried': []}
backtrack 换策略: found 最终策略 semantic


In [5]:
print("死磕关键词:", retry_result["status"], "尝试次数", len(retry_result["attempts"]))  # 死磕失败。
print("backtrack:", backtrack_result["status"], "最终策略", backtrack_result["strategy"])  # backtrack 成功。
print("换策略路径:", [name for name, _ in backtrack_result["log"]])  # 展示 keyword 到 semantic 换策略路径。

死磕关键词: not_found 尝试次数 3
backtrack: found 最终策略 semantic
换策略路径: ['keyword', 'semantic']


## 结果解读

死磕关键词检索三次都 not_found；backtrack 在检查点换策略，keyword 无命中后换 semantic 命中。要点：检查点打在分叉点、换策略排除已失败项、回退恢复状态一致性、backtrack 次数也要有预算。

In [6]:
assert retry_result["status"] == "not_found"  # 死磕同一策略始终失败。
assert len(retry_result["attempts"]) == 3  # 死磕重复了三次。
assert backtrack_result["status"] == "found"  # backtrack 换策略成功。
assert backtrack_result["strategy"] == "semantic"  # 最终由语义检索命中。
assert [n for n, _ in backtrack_result["log"]] == ["keyword", "semantic"]  # 换策略路径为 keyword 到 semantic。
print("全部不变量通过")  # 输出测试通过信号。

全部不变量通过
